In [ ]:
# --- 1. CONFIGURATION & FILE PATHS ---
# Using the specific files for CV (common) and CT/Q (frequency dependent)
cv_both_path = "Var_CV_char_Fboth_newvars_Tese_Varactor_characterization_tb1_Measurements_history_1_20260612_14_14_42.57.csv"
fmin_tb2_path = "Var_char_Fmin_newvar_Tese_Varactor_characterization_tb2_Measurements_history_1_20260612_14_15_30.35.csv"
fmax_tb2_path = "Var_char_Fmax_newvar_Tese_Varactor_characterization_tb2_Measurements_history_1_20260612_14_35_30.37.csv"


unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        multiplier = unit_map.get(match.group(2), 1.0)
        return num * multiplier
    return np.nan

# --- 2. LOADING & DATA CLEANING ---
df_cv = pd.read_csv(cv_both_path)
df_t_min = pd.read_csv(fmin_tb2_path)
df_t_max = pd.read_csv(fmax_tb2_path)

# Process CV Data
df_cv['Vdc'] = df_cv['Sweep:Variable:Vdc'].apply(parse_units)
df_cv['Cap'] = df_cv['Cap_measuredAt1Mhz_M7:ac'].apply(parse_units)
df_cv['Vopt_file'] = df_cv['deriv_Vdc_opt:ac'].apply(parse_units)

# Process Temperature Sweep Data
for df in [df_t_min, df_t_max]:
    df['Temp'] = df['Sweep:Variable:temp'].apply(parse_units)
    df['Cap'] = df['Cap_measuredAt1Mhz_M7:ac'].apply(parse_units)
    df['Q'] = df['Q_factorAt1Mhz:ac'].apply(parse_units)

corners = ['SS125', 'TT25', 'FF-40']
colors = {'SS125': 'red', 'TT25': 'green', 'FF-40': 'blue'}
display_names = {'SS125': 'SS 125°C', 'TT25': 'TT 25°C', 'FF-40': 'FF -40°C'}

# --- 3. PLOTTING ---
fig, axs = plt.subplots(3, 1, figsize=(10, 20))
plt.subplots_adjust(hspace=0.4)
summary_results = []

for corner in corners:
    c_label = display_names[corner]
    
    # 3.1 CV Curves (Overlapping fmin/fmax labels for style consistency)
    c_data = df_cv[df_cv['Corner'] == corner].sort_values('Vdc')
    if not c_data.empty:
        v = c_data['Vdc'].values
        cap_f = c_data['Cap'].values * 1e15
        
        # Plot fmin style (Solid)
        axs[0].plot(v, cap_f, color=colors[corner], linestyle='-', linewidth=2, label=f"{c_label} (fmin)")
        # Plot fmax style (Dashed)
        axs[0].plot(v, cap_f, color=colors[corner], linestyle='--', linewidth=2, label=f"{c_label} (fmax)")
        
        c_max, c_min = cap_f.max(), cap_f.min()
        v_opt_val = df_cv[df_cv['Corner'] == corner]['Vopt_file'].dropna()
        v_opt = v_opt_val.iloc[0] if not v_opt_val.empty else np.nan
        c_at_opt = np.nan
        
        if not np.isnan(v_opt):
            idx = (np.abs(v - v_opt)).argmin()
            c_at_opt = cap_f[idx]
            axs[0].scatter(v[idx], cap_f[idx], color='black', marker='x', s=100, zorder=5)
            axs[0].annotate(f'{v_opt*1000:.0f}mV', xy=(v[idx], cap_f[idx]), xytext=(10, 5), 
                            textcoords='offset points', fontweight='bold', color=colors[corner])

    # 3.2 CT & Q Overlap (Frequency dependent)
    t_min = df_t_min[df_t_min['Corner'] == corner].sort_values('Temp')
    t_max = df_t_max[df_t_max['Corner'] == corner].sort_values('Temp')
    
    # Capacitance vs Temp Plot
    axs[1].plot(t_min['Temp'], t_min['Cap']*1e15, color=colors[corner], linestyle='-', marker='o', label=f"{c_label} (fmin)")
    axs[1].plot(t_max['Temp'], t_max['Cap']*1e15, color=colors[corner], linestyle='--', marker='x', label=f"{c_label} (fmax)")
    
    # Q Factor vs Temp Plot
    axs[2].plot(t_min['Temp'], t_min['Q'], color=colors[corner], linestyle='-', marker='s', label=f"{c_label} (fmin)")
    axs[2].plot(t_max['Temp'], t_max['Q'], color=colors[corner], linestyle='--', marker='d', label=f"{c_label} (fmax)")

    # 3.3 Collect Summary Statistics
    # Finding Q at room temperature (~26°C in data)
    q_min_25 = t_min[np.isclose(t_min['Temp'], 26, atol=1.5)]['Q'].mean()
    q_max_25 = t_max[np.isclose(t_max['Temp'], 26, atol=1.5)]['Q'].mean()
    
    summary_results.append({
        'Corner': c_label, 'Cmax': c_max, 'Cmin': c_min, 'Ratio': c_max/c_min,
        'Vopt': v_opt * 1000, 'CatOpt': c_at_opt, 'Qmin': q_min_25, 'Qmax': q_max_25
    })

# Formatting Plots
titles = ["CV Curves Overlap (fmin vs fmax)", "Capacitance vs Temp Overlap", "Quality Factor vs Temp Overlap"]
for i, ax in enumerate(axs):
    ax.set_title(titles[i], fontweight='bold')
    ax.legend(prop={'size': 8}, ncol=2)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_ylabel("Capacitance (fF)" if i < 2 else "Quality Factor (Q)")
    ax.set_xlabel("Vdc (V)" if i == 0 else "Temperature (°C)")

plt.tight_layout()
plt.show()

# --- 4. PRINT POINTS OF INTEREST ---
print("\n" + "="*118)
print(f"{'Corner':<12} | {'Cmax (fF)':<10} | {'Cmin (fF)':<10} | {'Ratio':<8} | {'Vopt (mV)':<10} | {'C@Vopt (fF)':<12} | {'Qmin@26°C':<9} | {'Qmax@26°C':<9}")
print("-" * 118)
for res in summary_results:
    print(f"{res['Corner']:<12} | {res['Cmax']:>9.2f}  | {res['Cmin']:>9.2f}  | {res['Ratio']:>8.2f} | {res['Vopt']:>9.0f}  | {res['CatOpt']:>11.2f}  | {res['Qmin']:>9.1f} | {res['Qmax']:>9.1f}")
print("="*118)